In [1]:
SKIP_LLM_CHUNKING = True

## Ingestion


In [2]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data

In [3]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
evidently_docs = read_repo_data('evidentlyai', 'docs')

print(f"FAQ documents: {len(dtc_faq)}")
print(f"Evidently documents: {len(evidently_docs)}")

FAQ documents: 1285
Evidently documents: 95


## Chunking and Preprocessing

### Intelligent Chunking with LLM

In [4]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

# now you can access them
openai_key = os.getenv("OPENAI_API_KEY")
print("Has key?", bool(openai_key))

if not SKIP_LLM_CHUNKING:
    openai_client = OpenAI(openai_key)

    def llm(prompt, model='gpt-4o-mini'):
        messages = [
            {"role": "user", "content": prompt}
        ]

        response = openai_client.responses.create(
            model='gpt-4o-mini',
            input=messages
        )

        return response.output_text


Has key? True


In [5]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()


In [6]:
if not SKIP_LLM_CHUNKING:
    def intelligent_chunking(text):
        prompt = prompt_template.format(document=text)
        response = llm(prompt)
        sections = response.split('---')
        sections = [s.strip() for s in sections if s.strip()]
        return sections

In [7]:
from tqdm.auto import tqdm

if not SKIP_LLM_CHUNKING:
    evidently_chunks = []

    for doc in tqdm(evidently_docs):
        doc_copy = doc.copy()
        doc_content = doc_copy.pop('content')

        sections = intelligent_chunking(doc_content)
        for section in sections:
            section_doc = doc_copy.copy()
            section_doc['section'] = section
            evidently_chunks.append(section_doc)

### Simple Chunking

In [8]:
def sliding_window(seq, size, step):
  if size <= 0 or step <= 0:
    raise ValueError("size and step must be positive")

  n = len(seq)
  result = []

  for i in range(0, n, step):
    chunk = seq[i:i+size]
    result.append({'start': i, 'chunk': chunk})

    if i + size >= n:
      break
  
  return result

In [9]:
from tqdm.auto import tqdm

evidently_chunks = []
for doc in evidently_docs:
  doc_copy = doc.copy()
  doc_content = doc_copy.pop('content')
  chunks = sliding_window(doc_content, 2000, 1000)

  for chunk in chunks:
    chunk.update(doc_copy)

  evidently_chunks.extend(chunks)
  
print(f"Evidently chunks: {len(evidently_chunks)}")

Evidently chunks: 576


## Search

### Text search

In [10]:
from minsearch import Index

evidently_index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

evidently_index.fit(evidently_chunks)

In [11]:
query = 'What should be in a test dataset for AI evaluation?'
results = evidently_index.search(query)

print(f"Top result: {results[0]['chunk'][:500]}...")

Top result: Retrieval-Augmented Generation (RAG) systems rely on retrieving answers from a knowledge base before generating responses. To evaluate them effectively, you need a test dataset that reflects what the system *should* know.

Instead of manually creating test cases, you can generate them directly from your knowledge source, ensuring accurate and relevant ground truth data.

## Create a RAG test dataset

You can generate ground truth RAG dataset from your data source.

### 1. Create a Project

In th...


### Vector search

In [12]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [15]:
from minsearch import VectorSearch
from tqdm.auto import tqdm
import numpy as np

evidently_embeddings = []

for d in tqdm(evidently_chunks):
  v = embedding_model.encode(d['chunk'])
  evidently_embeddings.append(v)

evidently_embeddings = np.array(evidently_embeddings)

evidently_vindex = VectorSearch()
evidently_vindex.fit(evidently_embeddings, evidently_chunks)

  0%|          | 0/576 [00:00<?, ?it/s]

In [16]:
print(evidently_embeddings.shape)
print(len(evidently_chunks))

(576, 768)
576


In [17]:
query = 'What should be in a test dataset for AI evaluation?'
q = embedding_model.encode(query)
results = evidently_vindex.search(q)

print(f"Top result: {results[0]['chunk'][:500]}...")

Top result: When working on an AI system, you need test data to run automated evaluations for quality and safety. A test dataset is a structured set of test cases. It can contain:

* Just the inputs, or
* Both inputs and expected outputs (ground truth).

You can use this test dataset to:

* Run **experiments** and track if changes improve or degrade system performance.
* Run **regression testing** to ensure updates don’t break what was already working.
* **Stress-test** your system with complex or adversari...


### Hybrid search

In [ ]:
def pure_text_search(query, num_results):
  return evidently_index.search(query, num_results=num_results)

def vector_search(query, num_results):
  q = embedding_model.encode(query)
  return evidently_vindex.search(q, num_results=num_results)

def hybrid_search(query, num_results):
  text_results = pure_text_search(query, num_results)
  vector_results = vector_search(query, num_results)

  # Combine and deduplicate results
  seen_ids = set()
  combined_results = []

  for result in text_results + vector_results:
    if result['filename'] not in seen_ids:
      seen_ids.add(result['filename'])
      combined_results.append(result)

  return combined_results

## Agents and Tools

In [19]:
system_prompt = """
You are a helpful assistant about Evidently - an open-source Python library to evaluate, test, and monitor ML and LLM systems, from experiments to production.

Use the search tool to find relevant information before answering questions.

If you can find specific information through search, use it to provide accurate
answers.

If the search doesn't return relevant results, let the user know and provide
general guidance
"""

In [ ]:
from typing import List, Any
def text_search(query: str) -> List[Any]:
  """
  Perform a text-based search on the Evidently index.
  Args:
    query (str): The search query string.
  Returns:
    List[Any]: A list of up to 5 search results returned by the Evidently index.
  """
  return hybrid_search(query, num_results=5)

In [26]:
from pydantic_ai import Agent
from pydantic_ai import Agent

agent = Agent(
  name="evidently_agent",
  instructions=system_prompt,
  tools=[text_search],
  model='gpt-4o-mini'
)

c:\Projects\Courses\aihero-jm\.venv\Lib\site-packages\pydantic_ai\models\__init__.py:1280: DeprecationWarning: Specifying a model name without a provider prefix is deprecated. Instead of 'gpt-4o-mini', use 'openai:gpt-4o-mini'.
  provider_name, model_name = parse_model_id(model)


In [27]:
%load_ext dotenv
%dotenv .env

question = "What should be in a test dataset for AI evaluation?"
result = await agent.run(user_prompt=question)

print(result)

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


Traceback (most recent call last):
  File "c:\Projects\Courses\aihero-jm\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3699, in run_code
    await eval(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\joao.melga\AppData\Local\Temp\ipykernel_38972\641551575.py", line 5, in <module>
    result = await agent.run(user_prompt=question)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Projects\Courses\aihero-jm\.venv\Lib\site-packages\pydantic_ai\agent\abstract.py", line 330, in run
    node = await agent_run.next(node)  # pyright: ignore[reportArgumentType]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Projects\Courses\aihero-jm\.venv\Lib\site-packages\pydantic_ai\run.py", line 356, in next
    return await self._run_node_with_hooks(node, self._advance_graph)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Projects\Courses\aihero-jm\.venv\Lib\site-packages\pydantic_ai\run.py", line 282, in _run_node_with_hooks
 

In [23]:
print(result)

AgentRunResult(output="It seems you might be referring to a specific course related to Evidently or a similar topic. However, I don't have direct information on courses or enrollment details.\n\nI recommend checking the official website or platform where the course is hosted for specific enrollment information. If you have any questions related to the Evidently library itself, feel free to ask!")
